In [12]:
%pip install pandas


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 199 kB/s eta 0:00:01
     |████████████████████████████████| 506 kB 27.1 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 35.4 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [16]:
from huggingface_hub import snapshot_download
from pathlib import Path

folder = Path.home() / "Downloads" / "freehire_data"

snapshot_download(
    repo_id="istrelov/freehire-jobs",
    repo_type="dataset",
    allow_patterns="*.jsonl.gz",
    local_dir=str(folder)
)

print("All parts downloaded!")


Fetching 20 files:   0%|          | 0/20 [00:00<?, ?it/s]

All parts downloaded!


In [13]:
import pandas as pd
from pathlib import Path

path = (
    Path.home() / "Downloads" / "freehire_data"
    / "freehire_jobs_part_007.jsonl.gz"
)

df = pd.read_json(
    path,
    lines=True,
    compression="gzip",
    nrows=1000
)

df.head()

,public_slug,source,title,company,company_slug,location,description,url,posted_at,created_at,...,company_tagline,company_industries,company_hq_country,company_types,company_sizes,domains,yc_batch,yc_status,yc_stage,yc_flags
0,operations-management-trainee-center-tx-tyson-...,workday,"Operations Management Trainee- Center, TX",Tyson Foods,tyson-foods,"Center Processing Plant - Center, Texas",<p></p><p><b>Job Details:</b></p><p><b>Job Sum...,https://tysonfoods.wd5.myworkdayjobs.com/TSN/j...,2026-07-27 00:00:00+00:00,2026-07-06 16:17:43.973468+00:00,...,Tyson Foods is a global food company that spec...,"[Manufacturing, Consumer Goods, Food & Beverag...",None,"[government, inhouse, outstaff, product]",[1000+],"[climatetech, devtools, healthcare, hrtech, lo...",[],[],[],[]
1,maintenance-tech-camilla-ga-july-tyson-foods-4...,workday,"Maintenance Tech Camilla, GA July",Tyson Foods,tyson-foods,"Camilla Plant - Camilla, Georgia",<p></p><p><b>Job Details:</b></p><p>**Acceptin...,https://tysonfoods.wd5.myworkdayjobs.com/TSN/j...,2026-07-06 00:00:00+00:00,2026-07-06 16:17:44.153832+00:00,...,Tyson Foods is a global food company that spec...,"[Manufacturing, Consumer Goods, Food & Beverag...",None,"[government, inhouse, outstaff, product]",[1000+],"[climatetech, devtools, healthcare, hrtech, lo...",[],[],[],[]
2,mechatronics-robotics-technician-cushman-wakef...,workday,Mechatronics & Robotics Technician,Cushman & Wakefield,cushman-wakefield,Client Site - USA - CA - Simi Valley - East St...,<span><span><span><span><span><span><span><h1>...,https://cw.wd1.myworkdayjobs.com/External/job/...,2026-07-06 00:00:00+00:00,2026-07-06 16:18:11.660251+00:00,...,Cushman & Wakefield is a global commercial rea...,"[Real Estate, Commercial Real Estate, Real Est...",us,"[agency, government, inhouse, outsource, product]",[51-200],"[ai, devtools, gamedev, govtech, healthcare, h...",[],[],[],[]
3,rme-operator-cushman-wakefield-dyyuoiyn,workday,RME Operator,Cushman & Wakefield,cushman-wakefield,Client Site - USA - CA - Rialto - 1568 N LINDE...,<span><span><span><span><span><span><span><h1>...,https://cw.wd1.myworkdayjobs.com/External/job/...,2026-07-06 00:00:00+00:00,2026-07-06 16:18:11.709298+00:00,...,Cushman & Wakefield is a global commercial rea...,"[Real Estate, Commercial Real Estate, Real Est...",us,"[agency, government, inhouse, outsource, product]",[51-200],"[ai, devtools, gamedev, govtech, healthcare, h...",[],[],[],[]
4,program-manager-fwgsolutions-42qvuxkb,lever,Program Manager,fwgsolutions,fwgsolutions,"Panama City Beach, FL",<div>FWG Solutions is a rapidly growing system...,https://jobs.lever.co/fwgsolutions/b5f7f4ba-b1...,2026-07-06 18:06:05.164000+00:00,2026-07-06 18:06:40.577775+00:00,...,None,[],None,[],[],[],[],[],[],[]


In [17]:
print(df.columns.tolist())


['public_slug', 'source', 'title', 'company', 'company_slug', 'location', 'description', 'url', 'posted_at', 'created_at', 'updated_at', 'closed_at', 'countries', 'regions', 'work_mode', 'skills', 'cities', 'collections', 'is_tech', 'seniority', 'category', 'posting_language', 'employment_type', 'education_level', 'english_level', 'experience_years_min', 'salary_min', 'salary_max', 'salary_currency', 'salary_period', 'company_name', 'company_tagline', 'company_industries', 'company_hq_country', 'company_types', 'company_sizes', 'domains', 'yc_batch', 'yc_status', 'yc_stage', 'yc_flags']


columns don't need: description, created_at, updated_at, closed_at, collections, posting_language, education_level, english_level, company_name, company_tagline, ya_batch, yc_status, yc_stage, yc_flags

filter: is_tech = TRUE, countries = us


In [20]:
%pip install pyarrow


Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 31.2 MB 17.4 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import gzip
import json
from pathlib import Path
import pandas as pd

folder = Path.home() / "Downloads" / "freehire_data"

drop_cols = ["description", "created_at", "updated_at", "closed_at", "collections",
             "posting_language", "education_level", "english_level", "company_name",
             "company_tagline", "yc_batch", "yc_status", "yc_stage", "yc_flags"]

rows = []

for file in sorted(folder.glob("*.jsonl.gz")):
    print("Reading", file.name)
    text = ""
    for line in gzip.open(file, "rt", encoding="utf-8", newline="\n"):
        if line.startswith('{"'):   # a new job starts
            text = ""
        text += line
        try:
            row = json.loads(text, strict=False)
        except json.JSONDecodeError:
            continue                # job not finished yet, read next line
        if row.get("is_tech") == True:
            for col in drop_cols:
                row.pop(col, None)
            rows.append(row)

data = pd.DataFrame(rows)
data.to_csv(folder / "freehire_tech.csv", index=False)

Reading freehire_jobs_part_000.jsonl.gz
Reading freehire_jobs_part_001.jsonl.gz
Reading freehire_jobs_part_002.jsonl.gz
Reading freehire_jobs_part_003.jsonl.gz
Reading freehire_jobs_part_004.jsonl.gz
Reading freehire_jobs_part_005.jsonl.gz
Reading freehire_jobs_part_006.jsonl.gz
Reading freehire_jobs_part_007.jsonl.gz
Reading freehire_jobs_part_008.jsonl.gz
Reading freehire_jobs_part_009.jsonl.gz
Reading freehire_jobs_part_010.jsonl.gz
Reading freehire_jobs_part_011.jsonl.gz
Reading freehire_jobs_part_012.jsonl.gz
Reading freehire_jobs_part_013.jsonl.gz
Reading freehire_jobs_part_014.jsonl.gz
Reading freehire_jobs_part_015.jsonl.gz
Reading freehire_jobs_part_016.jsonl.gz
Reading freehire_jobs_part_017.jsonl.gz
Reading freehire_jobs_part_018.jsonl.gz
Reading freehire_jobs_part_019.jsonl.gz


ArrowKeyError: No type extension with name arrow.py_extension_type found

In [23]:
data = pd.read_csv("~/Downloads/freehire_data/freehire_tech.csv")

In [35]:
data["posted_at"] = pd.to_datetime(data["posted_at"], errors="coerce")
 
# Size
print("Rows:", len(data))
print("Columns:", data.shape[1])
print(list(data.columns))
 
# Duplicates
print("\n--- Duplicates ---")
print("Fully identical rows:", data.duplicated().sum())
print("Duplicate public_slug:", data["public_slug"].duplicated().sum())
print("Duplicate url:", data["url"].duplicated().sum())
print("Same title + company + location:", data.duplicated(["title", "company", "location"]).sum())
 
# Time range
print("\n--- Time range (posted_at) ---")
print("Earliest:", data["posted_at"].min())
print("Latest:  ", data["posted_at"].max())
print("Missing dates:", data["posted_at"].isna().sum())
print(data["posted_at"].dt.to_period("M").value_counts().sort_index())
 
# Missing values (% per column)
print("\n--- Missing values (%) ---")
print((data.isna().mean() * 100).round(1).sort_values(ascending=False))
 
# Main categories
for col in ["source", "countries", "category", "seniority", "work_mode", "employment_type", "salary_currency"]:
    print(f"\n--- Top 10 {col} ---")
    print(data[col].value_counts(dropna=False).head(10))
 
# Companies and salaries
print("\nUnique companies:", data["company"].nunique())
print(data["company"].value_counts().head(10))
print("\nRows with a salary:", data["salary_min"].notna().sum())
print(data[["salary_min", "salary_max", "experience_years_min"]].describe().round(0))



Rows: 876684
Columns: 27
['public_slug', 'source', 'title', 'company', 'company_slug', 'location', 'url', 'posted_at', 'countries', 'regions', 'work_mode', 'skills', 'cities', 'is_tech', 'seniority', 'category', 'employment_type', 'experience_years_min', 'salary_min', 'salary_max', 'salary_currency', 'salary_period', 'company_industries', 'company_hq_country', 'company_types', 'company_sizes', 'domains']

--- Duplicates ---
Fully identical rows: 0
Duplicate public_slug: 0
Duplicate url: 30575
Same title + company + location: 166473

--- Time range (posted_at) ---
Earliest: 1900-01-02 00:00:00+00:00
Latest:   2026-08-09 00:00:00+00:00
Missing dates: 253069


/var/folders/1f/qrjb0mpd1csb3rl31p25lpsw0000gn/T/ipykernel_32242/1122115733.py:20: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  print(data["posted_at"].dt.to_period("M").value_counts().sort_index())


posted_at
1900-01         2
1970-01         1
2009-12         1
2012-02         6
2012-03         2
            ...  
2026-04     26472
2026-05     48594
2026-06    158146
2026-07    271348
2026-08     56606
Freq: M, Name: count, Length: 178, dtype: int64

--- Missing values (%) ---
salary_max              84.3
salary_period           84.0
salary_min              83.9
salary_currency         81.4
company_hq_country      63.5
work_mode               61.2
employment_type         58.7
seniority               58.3
experience_years_min    40.2
posted_at               28.9
category                13.6
location                 2.1
domains                  1.6
company_types            1.6
company_slug             1.6
company                  1.6
company_sizes            1.6
company_industries       1.6
public_slug              0.0
source                   0.0
cities                   0.0
skills                   0.0
regions                  0.0
countries                0.0
url                 